In [16]:
import urllib.request
import pandas as pd
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download ", "and extraction.")
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)
        
    og_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(og_file_path, data_file_path)
    print(f"Downloaded and saved as{data_file_path}")
    
download_and_unzip(url, zip_path, extracted_path, data_file_path)

Downloaded and saved assms_spam_collection/SMSSpamCollection.tsv


In [18]:
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])

In [20]:
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [26]:
df["Label"].value_counts()["ham"]

4825

In [38]:
def balance_dataset(df):
    num_spam = min(df["Label"].value_counts()["ham"], df["Label"].value_counts()["spam"])
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    df_balanced = pd.concat([ham_subset, df[df["Label"]=="spam"]])
    return df_balanced
df = balance_dataset(df)
print(df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [50]:
df["Label"] = df["Label"].map({"ham": 0, "spam": 1})

In [48]:
def random_split(df, train_frac, val_frac):
    df = df.sample(frac = 1, random_state= 123).reset_index(drop = True)
    train_end = int(len(df) * train_frac)
    val_end = train_end + int(len(df) *val_frac)

    train_df = df[:train_end]
    val_df = df[train_end:val_end]
    test_df = df[val_end:]

    return train_df, val_df, test_df
    

In [52]:
train_df, val_df, test_df = random_split(df, .7, .1)

In [54]:
train_df.to_csv("train_df.csv", index=None)
val_df.to_csv("val_df.csv", index=None)
test_df.to_csv("test_df.csv", index=None)

In [58]:
#checking end of text token id to pad the messages so we can batch them
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2") 
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [ ]:
import torch
from torch.utils.data import Dataset

class Spamdataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length = None, pad_token_id = 50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]

        if max_length = None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [encoded_text[:self.max_length]
                                 for encoded_text in self.encoded_texts]

        self.encoded_texts = [encoded_text + [pad_token_id]*(self.max_length - len(encoded_text))
                             for encoded_text in encoded_texts]
    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype = torch.long)
            torch.tensor(label, dtype = torch.long)
        )
    def __len__(self):
        return len(self.data)
    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            
            if len(encoded_text) > max_length:
                max_length = len(encoded_text)
        return max_length